# Auxiliary dataset investigation

This notebook investigates which supplied auxiliary datasets are useful for the CS5228 HDB monthly-rent regression task.

From `01_eda.ipynb`, the main modelling considerations are:

- rent changes strongly over time, so apparent temporal relationships must be checked with chronological validation;
- location is important, making block coordinates and nearby amenities promising;
- train and test must receive identical feature construction;
- no feature may use `MONTHLY_RENT` directly.

The notebook evaluates data quality, join coverage, simple associations, leakage risk, and likely feature value. It does not train a final model.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Works whether the notebook is launched from the repository root or notebooks/.
ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data" / "train.csv").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not find data/train.csv from the current directory")

DATA = ROOT / "data"
AUX = DATA / "auxiliary"

paths = {
    "train": DATA / "train.csv",
    "test": DATA / "test.csv",
    "hdb_blocks": AUX / "sg-hdb-block.csv",
    "mrt": AUX / "sg-mrt-stations.csv",
    "schools": AUX / "sg-schools.csv",
    "malls": AUX / "sg-shopping-malls.csv",
    "stocks": AUX / "sg-stock-prices.csv",
    "coe": AUX / "sg-coe-prices.csv",
}

missing = [str(path) for path in paths.values() if not path.exists()]
assert not missing, f"Missing files: {missing}"

frames = {name: pd.read_csv(path) for name, path in paths.items()}
train = frames["train"]
test = frames["test"]
print(f"Train: {train.shape}; test: {test.shape}")

Train: (150000, 11); test: (50000, 10)


## 1. Inventory and data quality

A useful auxiliary dataset should have adequate coverage, stable keys or valid coordinates, limited missingness, and information that is available at prediction time.

In [2]:
def audit(name, frame):
    return {
        "dataset": name,
        "rows": len(frame),
        "columns": frame.shape[1],
        "duplicate_rows": int(frame.duplicated().sum()),
        "missing_cells": int(frame.isna().sum().sum()),
        "missing_pct": 100 * frame.isna().sum().sum() / frame.size,
    }

inventory = pd.DataFrame(audit(name, frame) for name, frame in frames.items())
display(inventory.style.format({"missing_pct": "{:.2f}%"}))

for name, frame in frames.items():
    print(f"\n{name}: {frame.columns.tolist()}")

,dataset,rows,columns,duplicate_rows,missing_cells,missing_pct
0,train,150000,11,425,0,0.00%
1,test,50000,10,5508,0,0.00%
2,hdb_blocks,9464,11,0,0,0.00%
3,mrt,243,8,0,0,0.00%
4,schools,337,14,0,0,0.00%
5,malls,113,3,0,0,0.00%
6,stocks,91642,9,0,510,0.06%
7,coe,539,8,0,0,0.00%



train: ['RENT_APPROVAL_DATE', 'TOWN', 'BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL', 'FLOOR_AREA_SQM', 'FURNISHED', 'LEASE_COMMENCE_DATE', 'FEE', 'MONTHLY_RENT']

test: ['RENT_APPROVAL_DATE', 'TOWN', 'BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL', 'FLOOR_AREA_SQM', 'FURNISHED', 'LEASE_COMMENCE_DATE', 'FEE']

hdb_blocks: ['TOWN', 'BLOCK', 'STREET', 'POSTAL_CODE', 'LATITUDE', 'LONGITUDE', 'MAX_FLOOR', 'YEAR_COMPLETED', 'SUBZONE', 'PLANNING_AREA', 'REGION']

mrt: ['CODE', 'NAME', 'STATUS', 'LATITUDE', 'LONGITUDE', 'SUBZONE', 'PLANNING_AREA', 'REGION']

schools: ['NAME', 'URL', 'STREET', 'POSTAL_CODE', 'MRT_STATIONS', 'BUS_LINES', 'PLANNING_AREA', 'REGION', 'TYPE_CODE', 'NATRUE_CODE', 'SESSION_CODE', 'MAINLEVEL_CODE', 'LATITUDE', 'LONGITUDE']

malls: ['NAME', 'LATITUDE', 'LONGITUDE']

stocks: ['NAME', 'SYMBOL', 'DATE', 'OPEN', 'HIGH', 'LOW', 'CLOSE', 'ADJUSTED_CLOSE', 'VOLUME']

coe: ['YEAR', 'CATEGORY', 'MONTH', 'ROUND', 'QUOTA_PREMIUM', 'PREVAILING_QUOTA_PREMIUM', 'QUOTA', 'BIDS_RECEIVED']

## 2. HDB block metadata

The block table can be joined directly to rental records by normalized town, block, and street. It supplies coordinates and neighbourhood/building attributes, so join coverage is the key usefulness check.

In [3]:
ADDRESS_KEYS = ["TOWN", "BLOCK", "STREET"]


def normalize_text(series):
    return (
        series.astype("string")
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
    )


def normalized_addresses(frame):
    result = frame.copy()
    for column in ADDRESS_KEYS:
        result[column] = normalize_text(result[column])
    return result


hdb = normalized_addresses(frames["hdb_blocks"])
hdb["LATITUDE"] = pd.to_numeric(hdb["LATITUDE"], errors="coerce")
hdb["LONGITUDE"] = pd.to_numeric(hdb["LONGITUDE"], errors="coerce")
hdb = hdb.sort_values(ADDRESS_KEYS).drop_duplicates(ADDRESS_KEYS, keep="first")

coverage_rows = []
for name, rental_data in {"train": train, "test": test}.items():
    rental_keys = normalized_addresses(rental_data)[ADDRESS_KEYS]
    joined = rental_keys.merge(
        hdb[ADDRESS_KEYS + ["LATITUDE", "LONGITUDE"]],
        on=ADDRESS_KEYS,
        how="left",
        validate="many_to_one",
    )
    coverage_rows.append({
        "split": name,
        "address_match_pct": 100 * joined["LATITUDE"].notna().mean(),
        "unmatched_rows": int(joined["LATITUDE"].isna().sum()),
    })

coverage = pd.DataFrame(coverage_rows)
display(coverage.style.format({"address_match_pct": "{:.2f}%"}))
print("Unique HDB blocks:", len(hdb))
print("HDB rows with valid coordinates:", int(hdb[["LATITUDE", "LONGITUDE"]].notna().all(axis=1).sum()))

,split,address_match_pct,unmatched_rows
0,train,100.00%,0
1,test,100.00%,0


Unique HDB blocks: 9464
HDB rows with valid coordinates: 9464


## 3. MRT stations, schools, and shopping malls

These tables support proximity features after the HDB address join. The supplied MRT status is a current snapshot and does not contain opening dates, so filtering on `STATUS == open` would leak stations that opened after early rental months. The baseline therefore uses a conservative static proxy: legacy MRT lines that were operating by January 2021, with known later infill and extension codes excluded. Schools and malls remain secondary ablation candidates because their snapshots can also change over time.


In [4]:
EARTH_RADIUS_KM = 6371.0088
PRE2021_MRT_PREFIXES = {"ns", "ew", "cg", "ne", "cc", "ce", "dt"}
POST2021_OR_FUTURE_CODES = {"ns6", "ne18", "cc30", "cc31", "cc32", "dt4", "dt36", "dt37"}


def clean_points(frame):
    result = frame.copy()
    result["LATITUDE"] = pd.to_numeric(result["LATITUDE"], errors="coerce")
    result["LONGITUDE"] = pd.to_numeric(result["LONGITUDE"], errors="coerce")
    result = result[
        result["LATITUDE"].between(-90, 90)
        & result["LONGITUDE"].between(-180, 180)
    ]
    return result.drop_duplicates(["NAME", "LATITUDE", "LONGITUDE"])


def conservative_pre2021_mrt(frame):
    result = frame.copy()
    result["CODE_CLEAN"] = normalize_text(result["CODE"])
    result["CODE_PREFIX"] = result["CODE_CLEAN"].str.extract(r"^([a-z]+)", expand=False)
    status = normalize_text(result["STATUS"])
    keep = (
        status.eq("open")
        & result["CODE_PREFIX"].isin(PRE2021_MRT_PREFIXES)
        & ~result["CODE_CLEAN"].isin(POST2021_OR_FUTURE_CODES)
    )
    return clean_points(result.loc[keep])


def nearest_distance_km(origins, destinations, chunk_size=1000):
    """Return nearest Haversine distance for each origin without a large full matrix."""
    origins_rad = np.radians(origins)
    destinations_rad = np.radians(destinations)
    nearest = np.full(len(origins), np.nan)

    valid = np.flatnonzero(np.isfinite(origins).all(axis=1))
    for start in range(0, len(valid), chunk_size):
        positions = valid[start:start + chunk_size]
        source = origins_rad[positions]
        dlat = destinations_rad[None, :, 0] - source[:, None, 0]
        dlon = destinations_rad[None, :, 1] - source[:, None, 1]
        a = (
            np.sin(dlat / 2) ** 2
            + np.cos(source[:, None, 0])
            * np.cos(destinations_rad[None, :, 0])
            * np.sin(dlon / 2) ** 2
        )
        distances = 2 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
        nearest[positions] = distances.min(axis=1)
    return nearest


amenities = {
    "MRT_PRE2021_PROXY": conservative_pre2021_mrt(frames["mrt"]),
    "SCHOOL": clean_points(frames["schools"]),
    "MALL": clean_points(frames["malls"]),
}

hdb_spatial = hdb[ADDRESS_KEYS + ["LATITUDE", "LONGITUDE"]].copy()
origin_coordinates = hdb_spatial[["LATITUDE", "LONGITUDE"]].to_numpy(float)

amenity_summary = []
for name, points in amenities.items():
    destination_coordinates = points[["LATITUDE", "LONGITUDE"]].to_numpy(float)
    column = f"NEAREST_{name}_KM"
    hdb_spatial[column] = nearest_distance_km(origin_coordinates, destination_coordinates)
    amenity_summary.append({
        "dataset": name.lower(),
        "usable_locations": len(points),
        "blocks_with_distance_pct": 100 * hdb_spatial[column].notna().mean(),
        "median_nearest_km": hdb_spatial[column].median(),
        "p90_nearest_km": hdb_spatial[column].quantile(0.9),
    })

amenity_summary = pd.DataFrame(amenity_summary)
display(amenity_summary.style.format({
    "blocks_with_distance_pct": "{:.2f}%",
    "median_nearest_km": "{:.2f}",
    "p90_nearest_km": "{:.2f}",
}))


,dataset,usable_locations,blocks_with_distance_pct,median_nearest_km,p90_nearest_km
0,mrt_pre2021_proxy,122,100.00%,0.69,1.34
1,school,337,100.00%,0.29,0.58
2,mall,113,100.00%,0.75,1.52


In [5]:
# Simple associations are descriptive, not causal; time, flat size, and town confound them.
train_location = normalized_addresses(train).merge(
    hdb_spatial,
    on=ADDRESS_KEYS,
    how="left",
    validate="many_to_one",
)

distance_columns = ["NEAREST_MRT_PRE2021_PROXY_KM", "NEAREST_SCHOOL_KM", "NEAREST_MALL_KM"]
spatial_association = (
    train_location[distance_columns + ["MONTHLY_RENT"]]
    .corr(method="spearman")["MONTHLY_RENT"]
    .drop("MONTHLY_RENT")
    .rename("spearman_with_rent")
    .to_frame()
)
display(spatial_association.style.format("{:.3f}"))

,spearman_with_rent
NEAREST_MRT_PRE2021_PROXY_KM,-0.085
NEAREST_SCHOOL_KM,-0.012
NEAREST_MALL_KM,-0.088


## 4. Stock and COE prices

These are temporal rather than property-level datasets. They may capture economic conditions, but rent already has a strong time trend. A high full-period correlation can therefore be spurious and must not be treated as proof of predictive value. Any final use should be tested with lagged features under chronological validation.

In [6]:
monthly_rent = (
    train.assign(MONTH=pd.to_datetime(train["RENT_APPROVAL_DATE"], format="%Y-%m"))
    .groupby("MONTH", as_index=False)["MONTHLY_RENT"]
    .median()
)

# Build a simple equal-weight stock index: normalize each symbol to its first close,
# then average normalized levels by month.
stocks = frames["stocks"].copy()
stocks["DATE"] = pd.to_datetime(stocks["DATE"], errors="raise")
stocks["CLOSE"] = pd.to_numeric(stocks["CLOSE"], errors="coerce")
stocks = stocks.sort_values(["SYMBOL", "DATE"])
stocks["NORMALIZED_CLOSE"] = stocks["CLOSE"] / stocks.groupby("SYMBOL")["CLOSE"].transform("first")
monthly_stocks = (
    stocks.assign(MONTH=stocks["DATE"].dt.to_period("M").dt.to_timestamp())
    .groupby("MONTH", as_index=False)["NORMALIZED_CLOSE"]
    .mean()
)

coe = frames["coe"].copy()
coe["MONTH"] = pd.to_datetime(
    coe["YEAR"].astype(str) + "-" + coe["MONTH"].astype(str),
    format="%Y-%b",
    errors="raise",
)
coe["QUOTA_PREMIUM_NUMERIC"] = pd.to_numeric(
    coe["QUOTA_PREMIUM"].astype(str).str.replace(r"[$,]", "", regex=True),
    errors="coerce",
)
monthly_coe = coe.groupby("MONTH", as_index=False)["QUOTA_PREMIUM_NUMERIC"].mean()

temporal = monthly_rent.merge(monthly_stocks, on="MONTH", how="left").merge(monthly_coe, on="MONTH", how="left")
temporal_summary = pd.DataFrame({
    "dataset": ["stocks", "coe"],
    "month_coverage_pct": [
        100 * temporal["NORMALIZED_CLOSE"].notna().mean(),
        100 * temporal["QUOTA_PREMIUM_NUMERIC"].notna().mean(),
    ],
    "spearman_with_monthly_median_rent": [
        temporal[["MONTHLY_RENT", "NORMALIZED_CLOSE"]].corr(method="spearman").iloc[0, 1],
        temporal[["MONTHLY_RENT", "QUOTA_PREMIUM_NUMERIC"]].corr(method="spearman").iloc[0, 1],
    ],
})
display(temporal_summary.style.format({
    "month_coverage_pct": "{:.2f}%",
    "spearman_with_monthly_median_rent": "{:.3f}",
}))
print("Rental months:", temporal["MONTH"].min().date(), "to", temporal["MONTH"].max().date())

,dataset,month_coverage_pct,spearman_with_monthly_median_rent
0,stocks,100.00%,-0.862
1,coe,100.00%,0.739


Rental months: 2021-01-01 to 2025-03-01


## 5. Recommendation

The priority follows predictive usefulness, coverage, and leakage risk:

1. **HDB block data is the first priority.** Normalized address keys provide complete train/test coverage and add coordinates, planning area, region, completion year, and maximum floor.
2. **MRT is conditional.** Current status is not historical. The baseline may use the conservative pre-2021 network proxy, but a production feature should use station opening dates and month-specific distances.
3. **Schools and malls are secondary ablations.** They offer plausible proximity signals, but they may overlap with HDB geography and their current snapshots can differ from earlier rental months.
4. **COE and stocks are experimental.** They are indirect market proxies that strongly overlap with calendar time. Lag them and retain them only if they improve later-period validation.

The main table evidence supports one decision: HDB has reliable coverage and low leakage risk. The amenity and market tables quantify candidates and their caveats; they do not establish predictive value.


In [7]:
# Final sanity checks for a reproducible investigation.
assert len(train_location) == len(train)
assert train_location[distance_columns].notna().all(axis=1).mean() == coverage.loc[
    coverage["split"].eq("train"), "address_match_pct"
].iloc[0] / 100
assert temporal["MONTH"].is_monotonic_increasing
assert set(distance_columns).issubset(train_location.columns)
print("All investigation checks passed.")

All investigation checks passed.
